# ENRICH-Aerial `nadir-2` — how far can **scop**'s bundle be thrown off and still come back?

[`ENRICH-Aerial_scop_sfm_gcp.ipynb`](ENRICH-Aerial_scop_sfm_gcp.ipynb) shows that
scop and pycolmap+Ceres agree on this block — check-point RMSE 1.21 cm against
1.25 cm, focal 5865.118 against 5865.136 px. But scop is handed COLMAP's
*converged* solution as its starting point, so that agreement only proves the two
engines share a stationary point. It does not prove scop can **find** it.

This notebook asks the other question: **degrade the initial values, keep the
observations bit-identical, and see whether LM returns to the same minimum.**

Because the observations never move, the minimum being searched for is literally
unchanged, and every difference is attributable to the optimiser rather than to a
different problem. That is the whole design of the experiment, and cell 4 below
proves it holds rather than asserting it.

## 0. What is perturbed, and what is not

Every scop `Observation` carries **two** scalars per variable `v`:

| attribute | role | read by |
|---|---|---|
| `obj.v` | the **observation** $l$ | `get_l()` (`Observation.py:317`, `Image.py:337`) |
| `obj._v_estimated` | the **unknown** — current/initial estimate | `Optimizer.to_jax_vector()` (`Optimizer.py:285`) |

For a control point `pt3.X` is the *surveyed* coordinate and `pt3._X_estimated` is
the unknown; for an image declared with `observations=["Xs","Ys","Zs"]`,
`img.Xs` is the *GNSS prior* and `img._Xs_estimated` is the pose being solved for.
`Point2D.x/y` are pure observations with no unknown at all.

So "perturb the initial values but not the observations" is exactly
`set_estimated_value` / `add_to_estimated_value`, and nothing else. That is all
`scop.utils.synthetic.perturb` does — the observation side is unreachable from it.

The one deliberate exception is `OBS_NOISE_PX` below, which adds white noise to
the image marks at **build** time. Leave it at 0 for the main experiment; turning
it on moves the minimum and the "recover the same solution" question stops being
well posed.

In [ ]:
import contextlib
import io
import time
import warnings
from pathlib import Path

import enrich_scop_block as blk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycolmap
from scop.utils.synthetic.perturb import (
    apply_similarity,
    assert_observations_unchanged,
    compare_to_snapshot,
    control_residuals,
    perturb_exterior,
    perturb_intrinsics,
    perturb_markers,
    perturb_tie_points,
    snapshot_observations,
    snapshot_solution,
)

import enrich_data

warnings.filterwarnings("ignore")

ROOT = Path(enrich_data.__file__).parents[1]
SPARSE = ROOT / "data/ENRICH-Aerial/nadir-2/scop/colmap_sparse_free"
SPARSE_REF = ROOT / "data/ENRICH-Aerial/nadir-2/scop/colmap_sparse_gcp"

MAX_POINTS = 5000  # None = full block (42 635 pts, ~15 s per trial instead of ~4 s)
MIN_TRACK_LENGTH = 3
OBS_NOISE_PX = 0.0  # > 0 perturbs the OBSERVATIONS; see the note above
SEEDS = [1, 2, 3]

RESULTS = None  # filled by the sweeps below

## 1. The block

`enrich_scop_block.build` is cells 2–17 of the tutorial notebook lifted into a
function — same Sim3-aligned free network, same `SimpleRadial(k=0)` camera with
only `f` freed, same per-axis GCP sigmas, same 0.5 px sigma for ties and marks.
Nothing here is new; see that notebook for why each piece looks the way it does.

In [ ]:
ctx = blk.build(
    sparse_path=SPARSE,
    min_track_length=MIN_TRACK_LENGTH,
    max_points=MAX_POINTS,
    obs_noise_px=OBS_NOISE_PX,
)
opt = ctx.opt

# The COLMAP free-network start, and the observation side we must never move.
x_start = opt.to_jax_vector()
obs_ref = snapshot_observations(opt)

## 2. The reference solution

The target is **scop's own converged solution**, not pycolmap's: comparing scop to
scop removes the engine-difference confound, so anything left over is the
optimiser's convergence behaviour. pycolmap is kept alongside as the external
sanity column.

`fit` is run tighter than in the tutorial — `epsilon=1e-8, conv_tol=1e-12` plus a
short Gauss-Newton polish. With the tutorial's default tolerances LM stops a few
millimetres short of the minimum, and that *stopping slop* (~8 mm) swamps the
effect being measured. Tightened, every run lands on the same point to ~0.015 mm,
which is the float32 floor (see §4).

In [ ]:
FIT_LM = dict(
    method="LM",
    iter_max=100,
    schur=True,
    compute_Qxx=False,
    verbose=False,
    epsilon=1e-8,
    conv_tol=1e-12,
)
FIT_GN = dict(
    method="GN",
    iter_max=3,
    schur=True,
    compute_Qxx=False,
    verbose=False,
    epsilon=1e-8,
    conv_tol=1e-12,
)


def fit():
    """LM to convergence, then a short GN polish.

    `_cached_identity_l` (`Optimizer.py:869`) caches the identity-observation
    values and is never invalidated, so it is popped before every call.

    Deliberately absent:
    - `robust_kernel` — `_robust_weights` assumes the identity rows are trailing
      (`Optimizer.py:1693`); here they are interleaved, so the weights land on
      the wrong residuals.
    - `disable_outliers` — it would change the *observation set* between trials
      and destroy the like-for-like comparison.
    - `retriangulate` — `retriangulate_all()` overwrites `Point3D.X/Y/Z` for every
      point, control included, i.e. it destroys the survey.
    """
    t0 = time.time()
    # `verbose=False` silences neither the banner nor the tqdm bar
    # (`Optimizer.py:1659` and `:1835`), and 90 trials of those is 900 kB of
    # notebook. Redirect both streams instead.
    sink = io.StringIO()
    with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
        opt.__dict__.pop("_cached_identity_l", None)
        opt.fit(**FIT_LM)
        n_lm = opt.nb_iterations
        opt.__dict__.pop("_cached_identity_l", None)
        opt.fit(**FIT_GN)
    return n_lm, time.time() - t0


n_lm, dt = fit()
REF = snapshot_solution(opt)
REF_S0 = float(opt.s0)
REF_TIE, _, REF_MRK, _ = ctx.reproj_rmse()
REF_CHK = ctx.marker_rmse(ctx.data.check_names)
REF_CTL = ctx.marker_rmse(ctx.data.control_names)
REF_F = ctx.cam.get_estimated_value("f")

assert_observations_unchanged(opt, obs_ref)  # the fit itself must not move l

print(f"reference fit: {n_lm} LM iterations, {dt:.1f} s")
print(f"  sigma0        {REF_S0:.6f}")
print(f"  tie reproj    {REF_TIE:.4f} px   marker {REF_MRK:.4f} px")
print(f"  check RMSE    {REF_CHK * 100:.3f} cm  <- TRUE ACCURACY")
print(f"  control RMSE  {REF_CTL * 100:.3f} cm")
print(f"  focal         {REF_F:.4f} px")

In [ ]:
# The external column: pycolmap + Ceres on the same data.
rec_ref = pycolmap.Reconstruction(str(SPARSE_REF))
print(
    f"pycolmap sparse_gcp: {rec_ref.num_images()} images, "
    f"{rec_ref.num_points3D()} points, focal {rec_ref.cameras[1].params[0]:.4f} px, "
    f"mean reproj {rec_ref.compute_mean_reprojection_error():.3f} px"
)

## 3. The guard

Everything downstream is meaningless unless the perturbations really do leave the
observations alone. So rather than trust it, check it: apply the most violent
perturbation in the notebook, then compare every `Point2D.x/y`, every
`Point3D.X/Y/Z` and every `Image.Xs/Ys/Zs` against the snapshot taken before the
reference fit. `assert_observations_unchanged` raises on the first scalar that
moved — bitwise, not within a tolerance.

In [ ]:
opt.from_jax_vector(x_start)
rng = np.random.default_rng(0)
perturb_exterior(opt, sigma_pos=50.0, sigma_rot_deg=20.0, rng=rng)
perturb_tie_points(opt, sigma=50.0, rng=rng)
perturb_markers(opt, sigma=50.0, rng=rng)
perturb_intrinsics(opt, rel_f=2.0)
apply_similarity(opt, t=(100.0, -50.0, 20.0), rot_deg=(5.0, 5.0, 5.0), ds=0.05)

assert_observations_unchanged(opt, obs_ref)
n_obs = (len(obs_ref["point2d"]), len(obs_ref["point3d"]), len(obs_ref["images"]))
print(
    f"OK: {n_obs[0]} image marks, {n_obs[1]} object points and {n_obs[2]} position "
    f"priors are bit-identical after the worst perturbation in this notebook"
)
print(f"    initial reprojection is now {ctx.reproj_rmse()[0]:,.0f} px")

opt.from_jax_vector(x_start)  # back to the COLMAP start

## 4. The trial harness

`from_jax_vector` writes only the *variables*, so resetting to `x_start` restores
the starting point without touching a single observation and without rebuilding
the problem — no `deepcopy`, no re-jit, ~4 s per trial at 5 000 points.

### Recovery criteria

Thresholds are set from the measured noise floor, not from wishful thinking. The
state vector is cast to **float32** in `to_jax_vector` (`Optimizer.py:295`); this
block lives in a local frame (x ∈ [−217, 69] m), so float32 resolves ~2·10⁻⁵ m =
**0.02 mm**, and that is exactly the spread seen between runs that did converge to
the same point. The thresholds below sit ~5× above it. On UTM-scale coordinates
the same experiment would fail on precision alone.

A run counts as **recovered** when all of:

| metric | threshold | floor observed |
|---|---|---|
| all values finite | — | — |
| \|Δσ̂₀\| | < 1e-4 | ~1e-6 |
| camera centres, RMS | < 0.1 mm | 0.015 mm |
| camera rotations, max | < 0.5″ | 0.03″ |
| \|Δf\| | < 0.01 px | 0.0002 px |
| object points, p95 | < 0.5 mm | 0.02 mm |
| \|Δ check RMSE\| | < 0.1 mm | 0.01 mm |

The finiteness test is not a formality: at large tie-point perturbations LM can
return normally having walked into an overflow, so `fit()` returning is *not*
evidence of a converged run.

In [ ]:
TOL = dict(s0=1e-4, pos_rms=1e-4, ang_max=0.5 / 3600, d_f=1e-2, pts_p95=5e-4, chk=1e-4)


def trial(
    label,
    seed=0,
    sigma_pos=0.0,
    sigma_rot_deg=0.0,
    sigma_tie=0.0,
    sigma_marker=0.0,
    rel_f=0.0,
    sim=None,
    family="",
):
    """One perturb -> fit -> compare cycle. Returns a row for the results table."""
    opt.from_jax_vector(x_start)
    rng = np.random.default_rng(seed)
    if sigma_pos or sigma_rot_deg:
        perturb_exterior(opt, sigma_pos=sigma_pos, sigma_rot_deg=sigma_rot_deg, rng=rng)
    if sigma_tie:
        perturb_tie_points(opt, sigma=sigma_tie, rng=rng)
    if sigma_marker:
        perturb_markers(opt, sigma=sigma_marker, rng=rng)
    if rel_f:
        perturb_intrinsics(opt, rel_f=rel_f)
    if sim:
        apply_similarity(opt, **sim)

    assert_observations_unchanged(opt, obs_ref)
    pre_px = ctx.reproj_rmse()[0]

    try:
        n_lm, dt = fit()
        d = compare_to_snapshot(opt, REF)
        tie, _, mrk, _ = ctx.reproj_rmse()
        row = dict(
            family=family,
            label=label,
            seed=seed,
            pre_px=pre_px,
            sigma_pos=sigma_pos,
            sigma_rot_deg=sigma_rot_deg,
            sigma_tie=sigma_tie,
            sigma_marker=sigma_marker,
            rel_f=rel_f,
            n_lm=n_lm,
            seconds=dt,
            s0=float(opt.s0),
            tie_px=tie,
            marker_px=mrk,
            d_s0=float(opt.s0) - REF_S0,
            pos_rms=d["pos_rms"],
            pos_max=d["pos_max"],
            ang_max=d["ang_max"],
            pts_p50=d["pts_p50"],
            pts_p95=d["pts_p95"],
            pts_max=d["pts_max"],
            d_f=d["cameras"][1]["f"],
            check_rmse=ctx.marker_rmse(ctx.data.check_names),
            control_rmse=ctx.marker_rmse(ctx.data.control_names),
        )
    except Exception as exc:  # a diverged run can also blow up inside the solver
        return dict(
            family=family,
            label=label,
            seed=seed,
            pre_px=pre_px,
            sigma_pos=sigma_pos,
            sigma_rot_deg=sigma_rot_deg,
            sigma_tie=sigma_tie,
            sigma_marker=sigma_marker,
            rel_f=rel_f,
            recovered=False,
            error=type(exc).__name__,
        )

    finite = np.isfinite([
        row["s0"],
        row["pos_rms"],
        row["pts_p95"],
        row["d_f"],
        row["check_rmse"],
    ]).all()
    row["recovered"] = bool(
        finite
        and abs(row["d_s0"]) < TOL["s0"]
        and row["pos_rms"] < TOL["pos_rms"]
        and row["ang_max"] < TOL["ang_max"]
        and abs(row["d_f"]) < TOL["d_f"]
        and row["pts_p95"] < TOL["pts_p95"]
        and abs(row["check_rmse"] - REF_CHK) < TOL["chk"]
    )
    row["error"] = ""
    return row


def show(rows):
    df = pd.DataFrame(rows)
    cols = [
        "label",
        "seed",
        "pre_px",
        "n_lm",
        "s0",
        "tie_px",
        "pos_rms",
        "ang_max",
        "d_f",
        "pts_p95",
        "check_rmse",
        "recovered",
    ]
    out = df.reindex(columns=cols).copy()
    out["pre_px"] = out["pre_px"].map(lambda v: f"{v:,.2f}" if v < 100 else f"{v:,.0f}")
    out["pos_rms"] = out["pos_rms"] * 1e3  # mm
    out["pts_p95"] = out["pts_p95"] * 1e3  # mm
    out["ang_max"] = out["ang_max"] * 3600  # arcsec
    out["check_rmse"] = out["check_rmse"] * 100  # cm
    return out.rename(
        columns={
            "pre_px": "pre [px]",
            "pos_rms": "dpos [mm]",
            "ang_max": 'drot ["]',
            "d_f": "df [px]",
            "pts_p95": "dpts p95 [mm]",
            "check_rmse": "check [cm]",
        }
    )

## 5. Sanity: the zero perturbation

The null trial must reproduce the reference **exactly** — not approximately. If it
does not, the reset path is leaking state and every number below is worthless.

In [ ]:
row = trial("L0 - no perturbation", seed=0)
assert row["pos_rms"] == 0.0 and row["pts_p95"] == 0.0, row
print(
    f"exact: dpos {row['pos_rms']} m, dpts {row['pts_p95']} m, "0
    f"ds0 {row['d_s0']:.2e}, recovered={row['recovered']}"
)

## 6. Choosing the levels — everything in pixels

The linearisation does not care about metres or degrees, it cares about how far
the observations are from where the current parameters predict them. So the
levels are set to give the four families comparable **image-space displacement**.

For this block: $f = 5865$ px, flying height $\approx 172.5$ m AGL,
GSD $\approx 2.94$ cm/px, 39 images over $287 \times 143$ m.

| perturbation | displacement | why |
|---|---|---|
| 1 m in $X_s/Y_s$ | ≈ 34 px | $f\,\delta/D$ |
| 1° in $\omega/\varphi/\kappa$ | ≈ 102 px | $f\,\delta$ |
| 1 m in tie $XYZ$ | ≈ 34 px | $f\,\delta/D$ |
| 1 % in $f$ | ≈ 36 px at the edge | $(r/f)\,\delta f$, $r \approx 3600$ px |

Six levels, each ~3× the last, from "sub-pixel" to "the block is unrecognisable":

| level | $\sigma_{pos}$ [m] | $\sigma_{rot}$ [°] | $\sigma_{tie}$ [m] | $\sigma_{mrk}$ [m] | $\Delta f$ | ≈ px |
|---|---|---|---|---|---|---|
| L1 | 0.1 | 0.02 | 0.1 | 0.05 | 0.1 % | 8 |
| L2 | 0.3 | 0.06 | 0.3 | 0.15 | 0.3 % | 23 |
| L3 | 1 | 0.2 | 1 | 0.5 | 1 % | 77 |
| L4 | 3 | 0.6 | 3 | 1.5 | 3 % | 235 |
| L5 | 10 | 2 | 10 | 5 | 10 % | 850 |
| L6 | 30 | 6 | 30 | 15 | 30 % | 4 600 |

For scale, L5 already displaces the projections by a quarter of the image width.

In [ ]:
LEVELS = {
    "L1": dict(
        sigma_pos=0.1, sigma_rot_deg=0.02, sigma_tie=0.1, sigma_marker=0.05, rel_f=0.001
    ),
    "L2": dict(
        sigma_pos=0.3, sigma_rot_deg=0.06, sigma_tie=0.3, sigma_marker=0.15, rel_f=0.003
    ),
    "L3": dict(
        sigma_pos=1.0, sigma_rot_deg=0.2, sigma_tie=1.0, sigma_marker=0.5, rel_f=0.01
    ),
    "L4": dict(
        sigma_pos=3.0, sigma_rot_deg=0.6, sigma_tie=3.0, sigma_marker=1.5, rel_f=0.03
    ),
    "L5": dict(
        sigma_pos=10.0, sigma_rot_deg=2.0, sigma_tie=10.0, sigma_marker=5.0, rel_f=0.10
    ),
    "L6": dict(
        sigma_pos=30.0, sigma_rot_deg=6.0, sigma_tie=30.0, sigma_marker=15.0, rel_f=0.30
    ),
}

rows_A = [trial("L0", seed=0, family="A")]
for name, kw in LEVELS.items():
    for seed in SEEDS:
        rows_A.append(trial(name, seed=seed, family="A", **kw))

show(rows_A)

## 7. One factor at a time — where does it actually break?

The joint ladder says the basin is wide but not *which* unknown gives out first.
Each family below moves one group and leaves the rest at the COLMAP start, pushed
until it fails.

Note what the markers row does *not* do: with 26 markers among thousands of tie
points, their initial values barely enter the normal equations, so even a 30 m
perturbation leaves the pre-fit reprojection essentially unchanged — the marker
initial value is not a meaningful risk on this network.

In [ ]:
FAMILIES = {
    "B - exterior position": [dict(sigma_pos=s) for s in (1, 3, 10, 30, 60, 120)],
    "C - exterior rotation": [dict(sigma_rot_deg=s) for s in (0.2, 1, 3, 6, 12, 25)],
    "D - object points": [dict(sigma_tie=s) for s in (1, 3, 10, 30, 60, 120)],
    "E - focal": [dict(rel_f=s) for s in (0.01, 0.1, 0.3, 0.6, -0.5, 2.0)],
    "F - markers": [dict(sigma_marker=s) for s in (0.5, 5, 30)],
}

rows_B = []
for family, grid in FAMILIES.items():
    for kw in grid:
        label = ", ".join(f"{k}={v:g}" for k, v in kw.items())
        for seed in SEEDS:
            rows_B.append(trial(label, seed=seed, family=family, **kw))

df_B = pd.DataFrame(rows_B)
(
    df_B
    .groupby(["family", "label"], sort=False)
    .agg(
        pre_px=("pre_px", "mean"),
        n_lm=("n_lm", "mean"),
        dpos_mm=("pos_rms", lambda s: 1e3 * s.mean()),
        dpts_p95_mm=("pts_p95", lambda s: 1e3 * s.mean()),
        recovered=("recovered", lambda s: f"{int(s.sum())}/{len(s)}"),
    )
    .style.format({
        "pre_px": "{:,.0f}",
        "n_lm": "{:.0f}",
        "dpos_mm": "{:.4f}",
        "dpts_p95_mm": "{:.4f}",
    })
)

## 8. The datum-invariant perturbation

A 7-parameter similarity applied to *all* initial values leaves the reprojection
residuals **exactly** invariant — the image observations cannot see it. Only the
datum constraints feel it: the 18 weighted control points and the 39 position
priors. There is no free-network or inner-constraint machinery in scop, so this
isolates whether that weighted datum alone pulls the block back.

The check in the first column is the point: `pre [px]` must stay at the reference
0.84 px however large the similarity, otherwise `apply_similarity` is wrong.

In [ ]:
SIMS = {
    "identity": dict(),
    "t = 0.5 m": dict(t=(0.5, 0.0, 0.0)),
    "t = 2 m": dict(t=(2.0, 0.0, 0.0)),
    "t = 10 m": dict(t=(10.0, 0.0, 0.0)),
    "t = 50 m": dict(t=(50.0, 0.0, 0.0)),
    "rot = 0.1 deg": dict(rot_deg=(0.0, 0.0, 0.1)),
    "rot = 1 deg": dict(rot_deg=(0.0, 0.0, 1.0)),
    "rot = 5 deg": dict(rot_deg=(0.0, 0.0, 5.0)),
    "ds = 1e-4": dict(ds=1e-4),
    "ds = 1e-3": dict(ds=1e-3),
    "ds = 1e-2": dict(ds=1e-2),
    "ds = 1e-1": dict(ds=1e-1),
    "combined": dict(t=(10.0, -5.0, 2.0), rot_deg=(1.0, 1.0, 1.0), ds=1e-2),
}

rows_S = [
    trial(name, seed=0, family="G - similarity", sim=kw) for name, kw in SIMS.items()
]
show(rows_S)

## 9. Results

In [ ]:
RESULTS = pd.concat(
    [pd.DataFrame(rows_A), df_B, pd.DataFrame(rows_S)], ignore_index=True
)
RESULTS.to_csv("results/perturbation_sweep.csv", index=False)
print(
    f"{len(RESULTS)} trials, {int(RESULTS['recovered'].sum())} recovered "
    f"-> results/perturbation_sweep.csv"
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
ok = RESULTS[RESULTS["recovered"] & RESULTS["pre_px"].gt(0)]

ax = axes[0]
for family, g in RESULTS[
    RESULTS["family"].str.startswith(("B", "C", "D", "E"))
].groupby("family"):
    g = g.groupby("pre_px")["recovered"].mean().sort_index()
    ax.plot(g.index, 100 * g.values, "o-", label=family)
ax.set_xscale("log")
ax.set_xlabel("initial reprojection error [px]")
ax.set_ylabel("recovered [%]")
ax.set_title("basin edge, one factor at a time")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(ok["pre_px"], 1e3 * ok["pos_rms"], s=18)
ax.axhline(1e3 * TOL["pos_rms"], color="C3", ls="--", lw=1, label="recovery threshold")
ax.axhline(0.02, color="grey", ls=":", lw=1, label="float32 floor")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("initial reprojection error [px]")
ax.set_ylabel("camera-centre RMS vs reference [mm]")
ax.set_title("how close the recovered runs land")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axes[2]
ax.scatter(ok["pre_px"], ok["n_lm"], s=18)
ax.set_xscale("log")
ax.set_xlabel("initial reprojection error [px]")
ax.set_ylabel("LM iterations")
ax.set_title("cost of getting back")
ax.grid(alpha=0.3)

fig.tight_layout()

## 10. What this says

**The basin is far wider than the block's geometry would suggest, and its edge is
sharp.** Every trial that stayed numerically finite came back to the reference
within **0.014 mm** on the camera centres, 0.03″ on the rotations, 0.0003 px on
the focal and 0.02 mm at the 95th percentile of the point cloud — i.e. to the
float32 resolution of the state vector, with σ̂₀ = 1.277746 and the tie
reprojection 0.8393 px reproduced identically. There is no intermediate regime in
which scop converges to a *nearby but different* solution: on this network it
either returns to the minimum or it overflows. All three seeds agree at every
level, so the edge is a property of the geometry, not of the draw.

| unknown | last level recovered | first level lost |
|---|---|---|
| camera position | **60 m** (4 056 px) | 120 m |
| camera rotation | **12°** (2 084 px) | 25° |
| object points | **30 m** (1 553 px) | 60 m |
| focal length | **3 × the true value** (4 098 px) | not reached |
| marker coordinates | 30 m — no effect at all | not reached |
| joint (L6: 30 m / 6° / 30 m / +30 % f) | **4 608 px** | — |

Reading the table:

- **The failures are geometric, not numerical.** They arrive at perturbations
  comparable to the 172 m flying height, where object points cross behind the
  cameras and the collinearity equation changes sign. No amount of damping
  recovers that, and it is the same wall any bundle would hit.
- **Focal length is the most forgiving unknown** — a 3× error still converges,
  because a wrong focal is a nearly pure radial scaling that the poses can absorb
  while LM walks `f` back. **Rotation is the least**, being the one term whose
  image cost is the full $f\delta$ with no range in the denominator.
- **Marker initial values are irrelevant** on a network of this redundancy: 26
  markers among 5 000 tie points do not move the pre-fit reprojection at all
  (0.8406 px at every σ, against 0.8393 px unperturbed), and their weighted
  survey observations pull them back regardless of where they start.
- **The datum handles a similarity on its own.** Every similarity trial converged,
  including a 50 m shift, a 5° rotation and a 10 % scale change. The `pre [px]`
  column stays at the reference 0.84 px throughout, which is the check that the
  perturbation really was datum-only: the image observations never see it, and it
  is undone by the 18 weighted control points and 39 position priors alone —
  there is no free-network or inner-constraint machinery involved.

### Two things to carry into other scop work

1. **`fit()` returning is not evidence of convergence.** At `sigma_tie = 60 m` LM
   exits normally with a state that has overflowed into values around 1e150; only
   the finiteness and σ̂₀ tests catch it. Any automated pipeline that trusts the
   absence of an exception will silently accept garbage.
2. **The tutorial's default tolerances stop LM about 8 mm short of the minimum.**
   That is invisible when the starting point is already converged — which is
   exactly the situation in the tutorial notebook — and dominant when it is not.
   With `epsilon=1e-8, conv_tol=1e-12` and a short GN polish the spread drops by
   almost three orders of magnitude, to the float32 floor.

### Caveat on precision

`to_jax_vector` casts the state to float32 (`Optimizer.py:295`). This block sits
in a local frame spanning ~300 m, so float32 resolves ~0.02 mm and everything
above is comfortably measurable. The same experiment on UTM-scale coordinates
would show ~1 m of quantisation and fail on precision alone, with nothing wrong
with the optimiser.